# Page-Level Document Boundary Detection on a Pharmaceutical PDF Bundle

**Document:** `pharma-blob-sample.pdf` (10 pages, several supplier documents bundled into one file)
**Stack:** PyMuPDF + Mistral-7B-Instruct (local, llama.cpp), with a Gemini cross-check
**Author:** Bashar

**Goal:** for every page, decide which document it belongs to and what type that document is, and output:

```json
[{"page": 0, "doc_id": 0, "doc_type": "Cover Letter"}, ...]
```

Three approaches are run and scored against the correct answer:

| | Approach | LLM calls |
|---|---|---|
| A | The course's prompts: "same document as the previous page? Yes/No", then "classify this page" | 1 per page + 1 per new document |
| B | A stricter version of the same pairwise prompt, with explicit rules | same as A |
| C | **The LLM describes each page on its own** (type + identifier, as JSON); **code** decides the boundaries | exactly 1 per page |

**Before running:** Runtime → Change runtime type → **T4 GPU** (for Mistral). Add a Colab secret `GOOGLE_API_KEY` to also run the Gemini cross-check at the end.

## Step 0: Install dependencies

In [ ]:
%pip install -q llama-cpp-python==0.3.19 --only-binary=llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
%pip install -q pymupdf google-genai huggingface_hub

## Step 1: Model and PDF

In [1]:
import os, glob, json, re, time
import llama_cpp
from llama_cpp import Llama

MODEL_PATH = os.environ.get("MISTRAL_GGUF", "/content/models/mistral-7b-instruct-v0.2.Q4_K_M.gguf")
if not os.path.exists(MODEL_PATH):
    from huggingface_hub import hf_hub_download
    hf_hub_download("TheBloke/Mistral-7B-Instruct-v0.2-GGUF", "mistral-7b-instruct-v0.2.Q4_K_M.gguf",
                    local_dir=os.path.dirname(MODEL_PATH))

PDF_PATH = os.environ.get("BLOB_PDF", "/content/pharma-blob-sample.pdf")
if not os.path.exists(PDF_PATH):
    # the course's sample bundle, shared publicly on Google Drive
    import urllib.request
    urllib.request.urlretrieve(
        "https://drive.google.com/uc?export=download&id=1fJ7DszcjB_TerDTAhXfAw74VqJBElTyw", PDF_PATH)
with open(PDF_PATH, "rb") as f:
    assert f.read(5) == b"%PDF-", "Download did not return a PDF; upload pharma-blob-sample.pdf to /content instead"

N_GPU_LAYERS = -1 if llama_cpp.llama_supports_gpu_offload() else 0
llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_gpu_layers=N_GPU_LAYERS, seed=42, verbose=False)
print(f"Mistral loaded on {'GPU' if N_GPU_LAYERS else 'CPU'}; PDF: {PDF_PATH}")

Mistral loaded on CPU; PDF: blob_dl/pharma-blob-sample.pdf


## Step 2: Extract the text of each page

The course uses PyPDF2. PyMuPDF is used here for consistency with the earlier notebooks; the result is the same list of `{"page_num", "text"}` dictionaries.

In [2]:
import fitz  # PyMuPDF

doc = fitz.open(PDF_PATH)
doc_pages = [{"page_num": i, "text": page.get_text()} for i, page in enumerate(doc)]
doc.close()

for p in doc_pages:
    first_lines = [l.strip() for l in p["text"].splitlines() if l.strip()][:2]
    print(f"page {p['page_num']}: {len(p['text'].split()):>3} words | {' / '.join(first_lines)}")

page 0: 191 words | Cytiva / 100 Results Way
page 1: 137 words | Certificate of Quality / This product is manufactured in compliance with our ISO 9001 certified quality management system.
page 2: 137 words | Certificate of Quality / This product is manufactured in compliance with our ISO 9001 certified quality management system.
page 3:  92 words | Packaging Component Specification / Document Number:
page 4: 101 words | Packaging Component Specification (continued) / Document Number: PKG-SPEC-2023-0847, Page 2 of 2
page 5: 151 words | Declaration Regarding Transmissible Spongiform / Encephalopathies (BSE/TSE Compliance Statement)
page 6: 156 words | Material Description Sheet / Product:
page 7: 130 words | Supplier Qualification Record / Supplier Name:
page 8: 128 words | Supplier Qualification Record (continued) / Supplier: Cytiva Sweden AB (SUP-2019-0847), Page 2 of 2
page 9: 150 words | Cytiva / 100 Results Way


## Step 3: The correct answer, for scoring

Worked out by reading every page. The bundle holds **8 documents in 10 pages**. Two details make it harder than it looks:

- **Pages 1 and 2 are two different Certificates of Quality** built from the same template (Low Flow Kit, lot 18356721, and High Flow Kit, lot 15102934). Same type, different documents.
- **Pages 4 and 8 are second pages** of the documents before them; each says "continued" and "Page 2 of 2".

In [3]:
import pandas as pd

DOC_TYPES = ["Cover Letter", "Certificate of Quality", "Packaging Specification", "BSE/TSE Declaration",
             "Material Description", "Supplier Qualification", "Chain of Custody", "Other"]

TRUTH = [  # (doc_id, doc_type) for pages 0-9
    (0, "Cover Letter"), (1, "Certificate of Quality"), (2, "Certificate of Quality"),
    (3, "Packaging Specification"), (3, "Packaging Specification"), (4, "BSE/TSE Declaration"),
    (5, "Material Description"), (6, "Supplier Qualification"), (6, "Supplier Qualification"),
    (7, "Chain of Custody"),
]

def starts_new(doc_ids):
    return [i == 0 or doc_ids[i] != doc_ids[i - 1] for i in range(len(doc_ids))]

def score(results, name):
    # Boundaries: does each page correctly start (or not start) a new document? Types: exact label match.
    pred_new = starts_new([r["doc_id"] for r in results])
    true_new = starts_new([t[0] for t in TRUTH])
    table = pd.DataFrame([{
        "page": r["page"], "doc_id": r["doc_id"], "doc_type": r["doc_type"],
        "correct type": TRUTH[i][1],
        "boundary ok": "yes" if pred_new[i] == true_new[i] else "NO",
        "type ok": "yes" if r["doc_type"] == TRUTH[i][1] else "NO",
    } for i, r in enumerate(results)]).set_index("page")
    summary = {"Approach": name,
               "Boundaries correct": f"{sum(p == t for p, t in zip(pred_new, true_new))}/{len(TRUTH)}",
               "Types correct": f"{sum(r['doc_type'] == t[1] for r, t in zip(results, TRUTH))}/{len(TRUTH)}",
               "Documents found": f"{results[-1]['doc_id'] + 1} (correct: 8)"}
    return table, summary

pd.DataFrame([{"page": i, "doc_id": d, "doc_type": t} for i, (d, t) in enumerate(TRUTH)]).set_index("page")

,doc_id,doc_type
page,,
0,0,Cover Letter
1,1,Certificate of Quality
2,2,Certificate of Quality
3,3,Packaging Specification
4,3,Packaging Specification
5,4,BSE/TSE Declaration
6,5,Material Description
7,6,Supplier Qualification
8,6,Supplier Qualification


## Step 4: One helper for calling the LLM

Temperature 0 so the same input always gives the same output. Mistral-Instruct's `[INST] ... [/INST]` prompt format is applied here, and every call is counted and timed.

In [4]:
stats = {"calls": 0, "seconds": 0.0}

def ask_mistral(prompt, max_tokens=60):
    t0 = time.perf_counter()
    out = llm(f"[INST] {prompt.strip()} [/INST]", max_tokens=max_tokens, temperature=0.0)
    stats["calls"] += 1
    stats["seconds"] += time.perf_counter() - t0
    return out["choices"][0]["text"].strip()

ask = ask_mistral  # swapped for Gemini in the cross-check at the end

def run_pairwise(is_same_document, classify_document_type):
    # The course's loop: compare each page with the previous one; classify whenever a new document starts
    results, current_doc_type, doc_counter = [], None, 0
    for i, page in enumerate(doc_pages):
        if i == 0:
            current_doc_type = classify_document_type(page["text"])
        else:
            same = is_same_document(doc_pages[i - 1]["text"], page["text"], current_doc_type)
            if not same:
                doc_counter += 1
                current_doc_type = classify_document_type(page["text"])
        results.append({"page": i, "doc_id": doc_counter, "doc_type": current_doc_type})
    return results

## Approach A: the course's prompts, unchanged

In [5]:
def is_same_document_course(prev_text, curr_text, doc_type=None):
    prompt = f'''
    You are checking whether two pages belong to the same document.
    Previous page type: {doc_type or 'unknown'}

    Previous Page:
    {prev_text}

    Current Page:
    {curr_text}

    Respond with only: Yes or No. Do not explain or add commentary.
    '''
    response = ask(prompt, max_tokens=5)
    return response.strip().lower().startswith("yes")

def classify_document_type_course(text):
    prompt = f'''
    This is the start of a new document. Based on the content, classify it.

    Page Content:
    {text}

    Choose from: Cover Letter, Certificate of Quality, Packaging Specification, BSE/TSE Declaration, Material Description, Supplier Qualification, Chain of Custody, Other.
    Just respond with the type.
    '''
    response = ask(prompt, max_tokens=15).strip().lower().replace(".", "")
    return response.title()

stats.update(calls=0, seconds=0.0)
results_a = run_pairwise(is_same_document_course, classify_document_type_course)
table_a, summary_a = score(results_a, "A: course prompts")
summary_a.update({"LLM calls": stats["calls"], "LLM seconds": round(stats["seconds"])})
print(summary_a)
table_a

{'Approach': 'A: course prompts', 'Boundaries correct': '7/10', 'Types correct': '4/10', 'Documents found': '5 (correct: 8)', 'LLM calls': 14, 'LLM seconds': 222}


,doc_id,doc_type,correct type,boundary ok,type ok
page,,,,,
0,0,This Document Is A Material Description Or Pro...,Cover Letter,yes,NO
1,0,This Document Is A Material Description Or Pro...,Certificate of Quality,NO,NO
2,1,Certificate Of Quality,Certificate of Quality,yes,NO
3,2,Packaging Specification,Packaging Specification,yes,yes
4,2,Packaging Specification,Packaging Specification,yes,yes
5,3,Bse/Tse Declaration,BSE/TSE Declaration,yes,NO
6,3,Bse/Tse Declaration,Material Description,NO,NO
7,4,Supplier Qualification,Supplier Qualification,yes,yes
8,4,Supplier Qualification,Supplier Qualification,yes,yes


## Approach B: a stricter pairwise prompt

Same loop, with rules added to the prompts to address A's mistakes:

- **Same-document check:** explicit rules for what counts as a continuation, and for when two same-type pages are still different documents.
- **Classifier:** a one-line definition of each type, and the answer must be one name from the list.

In [6]:
TYPE_DEFINITIONS = '''- Cover Letter: a letter addressed to the reader (greeting, sign-off), e.g. about storage or shipping
- Certificate of Quality: certifies one product lot (lot number, manufacture/expiry dates, release tests)
- Packaging Specification: packaging materials, configuration, packaging part numbers, drawings
- BSE/TSE Declaration: statement that products contain no animal-origin materials (BSE/TSE)
- Material Description: materials of construction and physical properties of a product
- Supplier Qualification: supplier approval status, audits, certifications, approved products
- Chain of Custody: where assemblies are manufactured and how they move to the customer
- Other: none of the above'''

def to_doc_type(text):
    # Map a free-text model answer onto the fixed list; anything unrecognised becomes "Other"
    exact = next((t for t in DOC_TYPES if t.lower() == text.strip().strip('".').lower()), None)
    return exact or next((t for t in DOC_TYPES if t.lower() in text.lower()), "Other")

def is_same_document_strict(prev_text, curr_text, doc_type=None):
    prompt = f'''
You are splitting a PDF that contains several separate documents bundled together.
Decide whether the CURRENT page continues the same document as the PREVIOUS page, or starts a new document.

Rules:
- CONTINUES if the current page says "continued", shows "Page 2 of 2" (or similar), or repeats the previous page's document number, supplier code or record title.
- NEW if the current page starts with its own document title or letterhead, or if its identifiers differ from the previous page.
- Two pages of the SAME TYPE are still SEPARATE documents if they have different lot numbers, product numbers or document numbers (for example two different certificates).

Previous page type: {doc_type or 'unknown'}

PREVIOUS PAGE:
{prev_text[:2500]}

CURRENT PAGE:
{curr_text[:2500]}

Answer with exactly one word: CONTINUES or NEW.
'''
    return ask(prompt, max_tokens=5).upper().startswith("CONT")

def classify_document_type_strict(text):
    prompt = f'''
Classify this page, which is the first page of a document, into exactly one document type.

Document types:
{TYPE_DEFINITIONS}

PAGE:
{text[:3000]}

Answer with the document type name only, exactly as written in the list.
'''
    return to_doc_type(ask(prompt, max_tokens=15))

stats.update(calls=0, seconds=0.0)
results_b = run_pairwise(is_same_document_strict, classify_document_type_strict)
table_b, summary_b = score(results_b, "B: stricter pairwise prompts")
summary_b.update({"LLM calls": stats["calls"], "LLM seconds": round(stats["seconds"])})
print(summary_b)
table_b

{'Approach': 'B: stricter pairwise prompts', 'Boundaries correct': '4/10', 'Types correct': '1/10', 'Documents found': '2 (correct: 8)', 'LLM calls': 11, 'LLM seconds': 220}


,doc_id,doc_type,correct type,boundary ok,type ok
page,,,,,
0,0,Material Description,Cover Letter,yes,NO
1,0,Material Description,Certificate of Quality,NO,NO
2,0,Material Description,Certificate of Quality,NO,NO
3,0,Material Description,Packaging Specification,NO,NO
4,0,Material Description,Packaging Specification,yes,NO
5,1,BSE/TSE Declaration,BSE/TSE Declaration,yes,yes
6,1,BSE/TSE Declaration,Material Description,NO,NO
7,1,BSE/TSE Declaration,Supplier Qualification,NO,NO
8,1,BSE/TSE Declaration,Supplier Qualification,yes,NO


## Approach C: describe each page, decide boundaries in code

Instead of asking the model to *compare* two pages, which A and B show it does badly, each page is described **on its own**. The model returns JSON with two facts:

- `doc_type`: one of the fixed types, using the same definitions as B
- `identifier`: the number that identifies *this specific* document (document number, lot number or record code), or empty

The boundary decision is then plain Python, so it is deterministic and easy to check:

1. A page that says **"continued"** or **"Page N of M" with N > 1** continues the previous document. This is detected with a regular expression, not the LLM.
2. Otherwise, a page starts a **new document** if its type differs from the previous page's, **or** both pages have identifiers and they differ (this is what separates the two certificates).
3. Otherwise it continues the previous document.

This also takes exactly one LLM call per page, fewer than A and B.

In [7]:
def describe_page(text):
    prompt = f'''
Read this page from a bundle of pharmaceutical supplier documents and describe it.

Document types:
{TYPE_DEFINITIONS}

PAGE:
{text[:3000]}

Return only a JSON object with these keys:
"doc_type": one document type name, exactly as written in the list above
"identifier": the main number that identifies this specific document (its document number, lot number or record code), or "" if there is none
'''
    raw = ask(prompt, max_tokens=80)
    match = re.search(r"\{.*\}", raw, re.S)
    try:
        data = json.loads(match.group(0)) if match else {}
    except json.JSONDecodeError:
        data = {}
    return {"doc_type": to_doc_type(str(data.get("doc_type", raw))),
            "identifier": str(data.get("identifier", "")).strip(), "raw": raw}

def is_continuation(text):
    # Explicit continuation markers near the top of the page
    head = text[:400].lower()
    m = re.search(r"page\s+(\d+)\s*(?:of|/)\s*(\d+)", head)
    return "continued" in head or bool(m and int(m.group(1)) > 1)

def run_per_page():
    page_info, results, doc_id = [], [], 0
    for i, page in enumerate(doc_pages):
        info = describe_page(page["text"])
        info["continuation"] = is_continuation(page["text"])
        if i > 0:
            prev = page_info[-1]
            type_changed = info["doc_type"] != prev["doc_type"]
            id_changed = bool(info["identifier"] and prev["identifier"] and info["identifier"] != prev["identifier"])
            if not info["continuation"] and (type_changed or id_changed):
                doc_id += 1
        # every page of a document carries the type assigned to its first page
        doc_type = results[-1]["doc_type"] if results and results[-1]["doc_id"] == doc_id else info["doc_type"]
        results.append({"page": i, "doc_id": doc_id, "doc_type": doc_type})
        page_info.append(info)
    return results, page_info

stats.update(calls=0, seconds=0.0)
results_c, page_info_c = run_per_page()
table_c, summary_c = score(results_c, "C: per-page description + rules")
summary_c.update({"LLM calls": stats["calls"], "LLM seconds": round(stats["seconds"])})
print(summary_c, "\n")
for i, info in enumerate(page_info_c):
    print(f"page {i}: {info['doc_type']:<24} identifier={info['identifier']!r:<24} continuation={info['continuation']}")
table_c

{'Approach': 'C: per-page description + rules', 'Boundaries correct': '10/10', 'Types correct': '10/10', 'Documents found': '8 (correct: 8)', 'LLM calls': 10, 'LLM seconds': 170} 

page 0: Cover Letter             identifier=''                       continuation=False
page 1: Certificate of Quality   identifier='18356721'               continuation=False
page 2: Certificate of Quality   identifier='15102934'               continuation=False
page 3: Packaging Specification  identifier='PKG-SPEC-2023-0847'     continuation=False
page 4: Packaging Specification  identifier='PKG-SPEC-2023-0847'     continuation=True
page 5: BSE/TSE Declaration      identifier='BSE-RA-2023-AKTA-001'   continuation=False
page 6: Material Description     identifier='29184612-AD'            continuation=False
page 7: Supplier Qualification   identifier='SUP-2019-0847'          continuation=False
page 8: Supplier Qualification   identifier='SUP-2019-0847'          continuation=True
page 9: Chain of Custody     

,doc_id,doc_type,correct type,boundary ok,type ok
page,,,,,
0,0,Cover Letter,Cover Letter,yes,yes
1,1,Certificate of Quality,Certificate of Quality,yes,yes
2,2,Certificate of Quality,Certificate of Quality,yes,yes
3,3,Packaging Specification,Packaging Specification,yes,yes
4,3,Packaging Specification,Packaging Specification,yes,yes
5,4,BSE/TSE Declaration,BSE/TSE Declaration,yes,yes
6,5,Material Description,Material Description,yes,yes
7,6,Supplier Qualification,Supplier Qualification,yes,yes
8,6,Supplier Qualification,Supplier Qualification,yes,yes


## Comparison

In [8]:
pd.DataFrame([summary_a, summary_b, summary_c]).set_index("Approach")

,Boundaries correct,Types correct,Documents found,LLM calls,LLM seconds
Approach,,,,,
A: course prompts,7/10,4/10,5 (correct: 8),14,222
B: stricter pairwise prompts,4/10,1/10,2 (correct: 8),11,220
C: per-page description + rules,10/10,10/10,8 (correct: 8),10,170


## Final output (Approach C)

In [9]:
final_json = json.dumps(results_c, indent=2)
with open("page_metadata.json", "w") as f:
    f.write(final_json)
print(final_json)

[
  {
    "page": 0,
    "doc_id": 0,
    "doc_type": "Cover Letter"
  },
  {
    "page": 1,
    "doc_id": 1,
    "doc_type": "Certificate of Quality"
  },
  {
    "page": 2,
    "doc_id": 2,
    "doc_type": "Certificate of Quality"
  },
  {
    "page": 3,
    "doc_id": 3,
    "doc_type": "Packaging Specification"
  },
  {
    "page": 4,
    "doc_id": 3,
    "doc_type": "Packaging Specification"
  },
  {
    "page": 5,
    "doc_id": 4,
    "doc_type": "BSE/TSE Declaration"
  },
  {
    "page": 6,
    "doc_id": 5,
    "doc_type": "Material Description"
  },
  {
    "page": 7,
    "doc_id": 6,
    "doc_type": "Supplier Qualification"
  },
  {
    "page": 8,
    "doc_id": 6,
    "doc_type": "Supplier Qualification"
  },
  {
    "page": 9,
    "doc_id": 7,
    "doc_type": "Chain of Custody"
  }
]


## Cross-check: Approach C with Gemini

The same prompt and rules with Gemini as the LLM, to check that C's result is not specific to one model. This makes 10 Gemini calls. It is skipped if no Gemini key is set.

In [10]:
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")

if GOOGLE_API_KEY:
    from google import genai
    client = genai.Client(api_key=GOOGLE_API_KEY)

    def ask_gemini(prompt, max_tokens=None, retries=5):
        # max_tokens is ignored: Gemini's hidden thinking tokens count against any output limit
        for attempt in range(retries):
            try:
                t0 = time.perf_counter()
                resp = client.models.generate_content(model="gemini-3.8-flash", contents=prompt.strip(),
                                                      config={"temperature": 0})
                stats["calls"] += 1
                stats["seconds"] += time.perf_counter() - t0
                return resp.text.strip()
            except Exception as e:
                # a used-up daily quota will not recover by retrying, so stop straight away
                if attempt == retries - 1 or "RESOURCE_EXHAUSTED" in str(e):
                    raise
                print(f"  Gemini call failed ({type(e).__name__}); retrying in {10 * (attempt + 1)}s")
                time.sleep(10 * (attempt + 1))

    ask = ask_gemini
    stats.update(calls=0, seconds=0.0)
    try:
        results_g, page_info_g = run_per_page()
    except Exception as e:
        results_g = None
        reason = "daily free-tier quota used up" if "RESOURCE_EXHAUSTED" in str(e) else type(e).__name__
        print(f"Gemini cross-check could not run ({reason}). Re-run this cell later.")
    finally:
        ask = ask_mistral
    if results_g:
        table_g, summary_g = score(results_g, "C with Gemini")
        summary_g.update({"LLM calls": stats["calls"], "LLM seconds": round(stats["seconds"])})
        print(summary_g, "\n")
        for i, (m, g) in enumerate(zip(page_info_c, page_info_g)):
            print(f"page {i}: Mistral {m['doc_type']} / {m['identifier']!r:<24} Gemini {g['doc_type']} / {g['identifier']!r}")
        print("\nSame final JSON as Mistral:", results_g == results_c)
else:
    print("No GOOGLE_API_KEY set; Gemini cross-check skipped.")

Gemini cross-check could not run (daily free-tier quota used up). Re-run this cell later.


## Findings

Results with Mistral-7B at temperature 0. They were identical across repeated runs. The LLM time is from a laptop CPU; on a Colab T4 GPU it is much shorter.

| Approach | Boundaries correct | Types correct | Documents found (correct: 8) | LLM calls | LLM time (CPU) |
|---|---|---|---|---|---|
| A: course prompts | 7/10 | 4/10 | 5 | 14 | 222 s |
| B: stricter pairwise prompts | 4/10 | 1/10 | 2 | 11 | 220 s |
| C: per-page description + rules | **10/10** | **10/10** | **8** | **10** | **170 s** |

**1. The course prompts merge documents and return unusable labels.** Approach A treated the first Certificate of Quality as part of the cover letter. It also merged the Material Description into the BSE/TSE Declaration, and the Chain of Custody list into the Supplier Qualification Record. The classifier often ignored the list of types: page 0 came back as "This Document Is A Material Description Or Product Information". Even correct answers were mangled by `.title()`, which turns "BSE/TSE Declaration" into "Bse/Tse Declaration" and "Certificate of Quality" into "Certificate Of Quality". Code that routes documents by type would not recognise either.

**2. A longer, stricter prompt made things worse.** Approach B added explicit rules for what counts as a continuation and for telling same-type documents apart. Mistral then answered "CONTINUES" for almost every page and found only 2 documents. For a 7B model, asking it to compare two full pages *and* apply several rules at once was too much; more instructions did not mean better answers. B's classifier did at least return valid labels, because its answers were checked against the fixed list.

**3. Approach C works by giving the model an easier question.** The model is never asked to compare pages. It is asked two things it can read straight off a single page: what type of document this is, and which number identifies it. The comparison is then done in Python:
- The two Certificates of Quality are separated because their identifiers differ (lots 18356721 and 15102934).
- Pages 4 and 8 are joined to the pages before them by two independent signals: the regular expression finds "continued" and "Page 2 of 2", and the identifiers match (PKG-SPEC-2023-0847, SUP-2019-0847).
- It also needs fewer LLM calls (exactly one per page) and every decision can be traced: the table above shows which identifier and continuation flag led to each split.

**4. Output checks matter as much as the prompt.** Every answer is parsed as JSON and mapped onto the fixed list of types, so a rambling answer becomes "Other" rather than a new, unexpected label. That keeps the metadata safe to route on.

**Limits.**
- This is one 10-page sample, and the correct answer was worked out by hand.
- The identifier rule would fail if a single document showed different identifiers on different pages, or if two same-type documents with no identifiers sat next to each other (they would be merged).
- The Gemini cross-check could not run because the free tier's 20 requests per day were already used; the cell can be re-run later.
